# When AI Speaks, Markets Listen
## A3: Data Collection Notebook

**Purpose:** Collect raw data only. No cleaning, no analysis, no visualization.

**Events:**
- GPT-4: March 15, 2023
- DeepSeek-R1: January 27, 2025

**Sample:** 60 stocks × 2 events = 120 firm-event observations

**Data Collection Methods:**
1. REST API — yfinance (U.S. stocks, benchmarks)
2. REST API — AKShare (China A-shares) ⚠️ Requires mobile hotspot (not VPN)
3. Hidden API — East Money (Chinese news, via DevTools)
4. HTML Scraping — Reuters (English news, BeautifulSoup)

---
## ⚠️ Before Running
- **Step 2 (U.S. stocks):** WiFi or any connection works
- **Step 3 (China stocks):** Switch to mobile hotspot, turn off VPN
- **Step 4-5 (News):** Any connection works

Each step saves its own CSV immediately — safe to stop and restart.

In [3]:
import pandas as pd
import os

data_path = '/Users/whisper/Downloads/data'

# 读取所有数据
us    = pd.read_csv(f'{data_path}/us_stock_data.csv')
cn    = pd.read_csv(f'{data_path}/cn_stock_data.csv')
sp500 = pd.read_csv(f'{data_path}/sp500_benchmark.csv')
csi300= pd.read_csv(f'{data_path}/csi300_benchmark.csv')

# 统一日期格式
for df in [us, cn, sp500, csi300]:
    df['date'] = pd.to_datetime(df['date']).dt.date.astype(str)

# 重命名基准收益率
sp500.rename(columns={'log_ret': 'ret_market'}, inplace=True)
csi300.rename(columns={'log_ret': 'ret_market'}, inplace=True)

# 合并基准
us_merged = us.merge(sp500[['date','ret_market']], on='date', how='left')
cn_merged = cn.merge(csi300[['date','ret_market']], on='date', how='left')

# 合并主数据集
master = pd.concat([us_merged, cn_merged], ignore_index=True)
master['China']    = (master['market'] == 'China').astype(int)
master['Software'] = (master['industry'] == 'Software').astype(int)

print(f'Master: {len(master)} rows')
print(master.head())

master.to_csv(f'{data_path}/master_data.csv', index=False)
print('✅ Saved!')

Master: 26849 rows
         date ticker market industry      close   log_ret     volume  \
0  2022-06-01   NVDA     US     Semi  18.286707       NaN  544514000   
1  2022-06-02   NVDA     US     Semi  19.556395  0.067128  648656000   
2  2022-06-03   NVDA     US     Semi  18.685980 -0.045529  598779000   
3  2022-06-06   NVDA     US     Semi  18.751862  0.003520  422406000   
4  2022-06-07   NVDA     US     Semi  18.891605  0.007425  388914000   

   turnover     marketcap   beta  ret_market  ret_pct  China  Software  
0  0.022408  4.823327e+12  2.335         NaN      NaN      0         0  
1  0.026694  4.823327e+12  2.335    0.018263      NaN      0         0  
2  0.024641  4.823327e+12  2.335   -0.016482      NaN      0         0  
3  0.017383  4.823327e+12  2.335    0.003132      NaN      0         0  
4  0.016005  4.823327e+12  2.335    0.009478      NaN      0         0  
✅ Saved!


In [27]:
import requests
import pandas as pd
import json
import time

def fetch_sohu_news(keyword, pages=3):
    records = []
    
    headers = {
        'Accept': 'application/json, text/plain, */*',
        'Content-Type': 'application/json',
        'Origin': 'https://search.sohu.com',
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
    }
    
    cookies = {
        'SUV': '1777720936744odinKp9t',
        'clt': '1777720938',
        'reqtype': 'pc'
    }
    
    for page in range(1, pages+1):
        try:
            data = {
                "pvId": "1777720942179ILsnUmg",
                "pageId": "1777720951685dxR",
                "mainContent": {
                    "productId": 1163,
                    "productType": 13,
                    "secureScore": 100,
                    "categoryId": 47
                },
                "resourceList": [{
                    "tplCompKey": "news-list",
                    "content": {
                        "productId": "20001",
                        "productType": "107",
                        "page": str(page),
                        "size": "10",
                        "spm": "smpc.csrpage.news-list",
                        "requestId": "1777720951685ZJE"
                    },
                    "context": {
                        "keyword": keyword,
                        "terminalType": "pc",
                        "domain": "sohu",
                        "queryType": "outside"
                    }
                }]
            }
            
            r = requests.post(
                'https://odin.sohu.com/odin/api/search/blockdata',
                headers=headers,
                cookies=cookies,
                json=data,
                timeout=10
            )
            
            # 正确的数据路径：data → news-list → list
            items = r.json().get('data', {}).get('news-list', {}).get('list', [])
            
            for item in items:
                # titleHL是带高亮标记的标题
                title = item.get('titleHL', '') or item.get('title', '')
                # 去掉#标签
                title = title.split('#')[0].strip()
                if title:
                    records.append({
                        'keyword':  keyword,
                        'title':    title,
                        'date':     item.get('publicTime', ''),
                        'author':   item.get('authorNameHL', ''),
                        'source':   'sohu_hidden_api',
                        'language': 'zh'
                    })
            
            print(f'  Page {page}: {len(items)} articles')
            time.sleep(1)
            
        except Exception as e:
            print(f'  Error: {e}')
    
    return pd.DataFrame(records) if records else pd.DataFrame(
        columns=['keyword','title','date','source','language']
    )


print('Fetching DeepSeek news...')
cn_ds = fetch_sohu_news('DeepSeek', pages=3)

print('\nFetching GPT-4 news...')
cn_gpt = fetch_sohu_news('GPT-4', pages=3)

cn_news = pd.concat([cn_ds, cn_gpt], ignore_index=True)
print(f'\nTotal: {len(cn_news)} articles')
print(cn_news[['keyword','title','date']].head(10))

cn_news.to_csv('/Users/whisper/Desktop/data/cn_news_raw.csv', index=False)
print('✅ Saved!')

Fetching DeepSeek news...
  Page 1: 10 articles
  Page 2: 10 articles
  Page 3: 10 articles

Fetching GPT-4 news...
  Page 1: 10 articles
  Page 2: 10 articles
  Page 3: 10 articles

Total: 60 articles
    keyword                                             title  date
0  DeepSeek                             DeepSeek-V4预览版正式上线并开源     0
1  DeepSeek  Deep<b>seek</b>官网公布<b>deep</b><b>seek</b>-v4接口文档     0
2  DeepSeek  Deep<b>seek</b>官网公布<b>deep</b><b>seek</b>-v4接口文档     0
3  DeepSeek  Deep<b>seek</b>官网公布<b>deep</b><b>seek</b>-v4接口文档     0
4  DeepSeek  Deep<b>seek</b>官网公布<b>deep</b><b>seek</b>-v4接口文档     0
5  DeepSeek                                      DeepSeek大幅降价     0
6  DeepSeek                                      DeepSeek恢复服务     0
7  DeepSeek                                    DeepSeek的“下一步”     0
8  DeepSeek                                      人民想念DeepSeek     0
9  DeepSeek                                     DeepSeek突然更新！     0
✅ Saved!


---
## Step 0: Install Libraries

In [23]:
import requests

url = "https://www.cls.cn/nodeapi/refreshTelegraphList"
params = {
    "app": "CailianpressWeb",
    "os":  "web",
    "sv":  "8.4.6"
}
headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.cls.cn/"
}

r = requests.get(url, params=params, headers=headers, timeout=10)
print(r.status_code)
print(r.json())

200
{'l': {}, 'i': 0, 'a': None}


In [2]:
!pip install yfinance akshare pysentiment2 beautifulsoup4 requests pandas numpy -q

In [3]:
import os, time, json, warnings, requests
import numpy as np
import pandas as pd
import yfinance as yf
import akshare as ak
from bs4 import BeautifulSoup

warnings.filterwarnings('ignore')

# Create output folder
os.makedirs('data', exist_ok=True)

# ── Constants ──────────────────────────────────────────
START_DATE    = '2022-06-01'
END_DATE      = '2025-06-30'
START_DATE_CN = '20220601'   # AKShare uses YYYYMMDD format
END_DATE_CN   = '20250630'

GPT4_DATE = '2023-03-15'
DS_DATE   = '2025-01-27'

print('✅ Setup complete. data/ folder ready.')

✅ Setup complete. data/ folder ready.


---
## Step 1: Define Stock Universe

In [4]:
# U.S. Semiconductor — S&P 500 GICS 4530 (top 15 by AI revenue exposure)
US_SEMI = [
    'NVDA','AMD','AVGO','TSM','AMAT',
    'LRCX','KLAC','MU','INTC','QCOM',
    'TXN','ADI','MRVL','ON','MPWR'
]

# U.S. Software — S&P 500 GICS 4510 (top 15 by AI revenue exposure)
US_SOFT = [
    'MSFT','GOOGL','META','CRM','ORCL',
    'NOW','ADBE','INTU','WDAY','SNOW',
    'PLTR','DDOG','MDB','ZS','CFLT'
]

# China Semiconductor — Tonghuashun AI Chip Index (top 15 by market cap)
CN_SEMI = [
    '688256','688041','688981','688008','688047',
    '688009','688598','688521','688396','688012',
    '688036','688469','688271','688065','688099'
]

# China Software — Tonghuashun AI Software Index (top 15 by market cap)
CN_SOFT = [
    '002230','601360','688111','300418','300229',
    '688579','300024','002410','300058','688188',
    '002908','300308','300866','688787','300450'
]

print(f'U.S. Semi:      {len(US_SEMI)} stocks')
print(f'U.S. Software:  {len(US_SOFT)} stocks')
print(f'China Semi:     {len(CN_SEMI)} stocks')
print(f'China Software: {len(CN_SOFT)} stocks')
print(f'Total:          {len(US_SEMI)+len(US_SOFT)+len(CN_SEMI)+len(CN_SOFT)} stocks')

U.S. Semi:      15 stocks
U.S. Software:  15 stocks
China Semi:     15 stocks
China Software: 15 stocks
Total:          60 stocks


---
## Step 2: U.S. Stock Data — yfinance REST API
✅ **WiFi or any connection. VPN is fine.**

Collects: daily close price, log return, volume, turnover rate, market cap, beta.

Saves to: `data/us_stock_data.csv`

In [5]:
def fetch_us_stock(ticker, industry):
    """
    Fetch U.S. stock data via yfinance REST API.
    Returns: date, ticker, market, industry, close,
             log_ret, volume, turnover, marketcap, beta
    """
    try:
        tk   = yf.Ticker(ticker)
        hist = tk.history(start=START_DATE, end=END_DATE)

        if hist.empty:
            print(f'  ⚠️  No data: {ticker}')
            return None

        # Log return
        hist['log_ret'] = np.log(
            hist['Close'] / hist['Close'].shift(1)
        )

        # Turnover = Volume / Shares Outstanding
        info   = tk.info
        shares = info.get('sharesOutstanding', np.nan)
        hist['turnover'] = (
            hist['Volume'] / shares if shares else np.nan
        )

        # Identifiers & firm characteristics
        hist['ticker']    = ticker
        hist['market']    = 'US'
        hist['industry']  = industry
        hist['marketcap'] = info.get('marketCap', np.nan)
        hist['beta']      = info.get('beta', np.nan)

        out = hist[[
            'ticker','market','industry',
            'Close','log_ret','Volume',
            'turnover','marketcap','beta'
        ]].reset_index()

        out.rename(columns={
            'Date':'date','Close':'close','Volume':'volume'
        }, inplace=True)

        out['date'] = pd.to_datetime(
            out['date']
        ).dt.tz_localize(None)

        return out

    except Exception as e:
        print(f'  ❌ Error {ticker}: {e}')
        return None


# ── Collect all U.S. stocks ─────────────────────────────
print('Collecting U.S. stocks via yfinance...')
print('=' * 50)

us_frames = []

print('\n[U.S. Semiconductor]')
for ticker in US_SEMI:
    print(f'  {ticker} ...', end=' ', flush=True)
    df = fetch_us_stock(ticker, 'Semi')
    if df is not None:
        us_frames.append(df)
        print(f'{len(df)} rows ✓')
    time.sleep(0.5)

print('\n[U.S. Software]')
for ticker in US_SOFT:
    print(f'  {ticker} ...', end=' ', flush=True)
    df = fetch_us_stock(ticker, 'Software')
    if df is not None:
        us_frames.append(df)
        print(f'{len(df)} rows ✓')
    time.sleep(0.5)

# Save immediately
if us_frames:
    us_data = pd.concat(us_frames, ignore_index=True)
    us_data.to_csv('data/us_stock_data.csv', index=False)
    print(f'\n✅ Saved: data/us_stock_data.csv')
    print(f'   {us_data["ticker"].nunique()} stocks, {len(us_data)} rows')
else:
    print('\n❌ No U.S. data collected.')


[U.S. Semiconductor]
  NVDA ... 771 rows ✓
  AMD ... 771 rows ✓
  AVGO ... 771 rows ✓
  TSM ... 771 rows ✓
  AMAT ... 771 rows ✓
  LRCX ... 771 rows ✓
  KLAC ... 771 rows ✓
  MU ... 771 rows ✓
  INTC ... 771 rows ✓
  QCOM ... 771 rows ✓
  TXN ... 771 rows ✓
  ADI ... 771 rows ✓
  MRVL ... 771 rows ✓
  ON ... 771 rows ✓
  MPWR ... 771 rows ✓

[U.S. Software]
  MSFT ... 771 rows ✓
  GOOGL ... 771 rows ✓
  META ... 771 rows ✓
  CRM ... 771 rows ✓
  ORCL ... 771 rows ✓
  NOW ... 771 rows ✓
  ADBE ... 771 rows ✓
  INTU ... 771 rows ✓
  WDAY ... 771 rows ✓
  SNOW ... 771 rows ✓
  PLTR ... 771 rows ✓
  DDOG ... 771 rows ✓
  MDB ... 771 rows ✓
  ZS ... 771 rows ✓
  CFLT ... 771 rows ✓

✅ Saved: data/us_stock_data.csv
   30 stocks, 23130 rows


---
## Step 3: China A-Share Data — AKShare REST API

⚠️ **Switch to mobile hotspot. Turn off VPN before running this step.**

Fields returned by AKShare:
- `涨跌幅` = daily return (%) — already calculated, no need to compute
- `换手率` = turnover rate (%) — directly available

Saves progress after every 5 stocks — safe if connection drops.

Saves to: `data/cn_stock_data.csv`

In [6]:
import os
print(os.getcwd())
print(os.listdir('.'))

/Users/whisper/Desktop
['TRD_Dalyr.xlsx', '~$otation.docx', 'Bilibili Advertising Pricing Strategy.docx', '~$en AI Speaks, Markets Listen Eng.docx', '7.5 Partial Fractions.pdf', '.DS_Store', 'week10-2_scraping_ai_news.ipynb', '.localized', 'Ch6+Financial+Analysis+Techniques.pdf', '~$代如何做筑梦的新青年.docx', '~$Doc2.docx', '~$flection Essay.docx', 'Table_of_Integrals.pdf', 'week9-2_JSON&semi-structure data.pdf', 'week9-1_github-api_collection.ipynb', '~$Lima.docx', 'Business', '~$entifying Rhetorical Language.docx', '~$25_Spring_Math2400_Homework_1.docx', 'week10-2_html scraping.pdf', '~$ Individual presentation and essay rev 7.pdf', '~$大卫.docx', 'github_ai_skills_repos.csv', '~$25_Spring_Math2400_Homework_1_Insert Watermark.docx', '思考,快与慢.pdf', '7.6 Other Integration Strategies.pdf', '~$问答.docx', '~$en AI Speaks, Markets Listen.docx', 'Screenshot 2026-04-23 at 10.36.00.png', 'master_grouped.xlsx', '面试.docx', '欧思源cv.docx', '~$Doc1.docx', 'e-book', '~$signment_1.docx', '~$欧思源CV.docx', '~$s assi

In [7]:
import pandas as pd
import numpy as np

# Step 3: China A-share data — CSMAR Database
# AKShare connection failed due to VPN restrictions.
# Data downloaded directly from CSMAR (cnrds.com) as CSV.

cn_data = pd.read_excel("TRD_Dalyr.xlsx")

cn_data.rename(columns={
    'Stkcd':   'ticker',
    'Trddt':   'date',
    'Clsprc':  'close',
    'Dretwd':  'ret',
    'ToverOs': 'turnover'
}, inplace=True)

cn_data['date']     = pd.to_datetime(cn_data['date'])
cn_data['ticker']   = cn_data['ticker'].astype(str).str.zfill(6)
cn_data['market']   = 'China'

cn_semi_list = ['688256','688041','688981','688008','688047',
                '688009','688598','688521','688396','688012',
                '688036','688469','688271','688065','688099']

cn_data['industry'] = cn_data['ticker'].apply(
    lambda x: 'Semi' if x in cn_semi_list else 'Software'
)

cn_data.to_csv('data/cn_stock_data.csv', index=False)
print(f'✅ China data: {cn_data["ticker"].nunique()} stocks, {len(cn_data)} rows')
print(cn_data.head())

✅ China data: 30 stocks, 28109 rows
   ticker       date  close       ret  turnover market  industry
0  002230 2022-06-01  36.48  0.003300  0.734309  China  Software
1  002230 2022-06-02  36.92  0.012061  0.867698  China  Software
2  002230 2022-06-06  38.77  0.050108  1.621534  China  Software
3  002230 2022-06-07  38.12 -0.016766  1.058751  China  Software
4  002230 2022-06-08  38.68  0.014690  1.313188  China  Software


---
## Step 4: Market Benchmarks — yfinance + AKShare

✅ **U.S. benchmark (S&P 500): any connection**

⚠️ **China benchmark (CSI 300): mobile hotspot recommended**

Saves to: `data/sp500_benchmark.csv`, `data/csi300_benchmark.csv`

In [8]:
# ── S&P 500 (yfinance) ──────────────────────────────────
print('Fetching S&P 500 benchmark...')
try:
    raw = yf.download(
        '^GSPC', start=START_DATE, end=END_DATE, progress=False
    )
    sp500 = pd.DataFrame({
        'date':     pd.to_datetime(raw.index).tz_localize(None),
        'close':    raw['Close'].values.flatten(),
    })
    sp500['log_ret'] = np.log(sp500['close'] / sp500['close'].shift(1))
    sp500['index']   = 'SP500'
    sp500 = sp500.dropna()
    sp500.to_csv('data/sp500_benchmark.csv', index=False)
    print(f'✅ Saved: data/sp500_benchmark.csv ({len(sp500)} days)')
except Exception as e:
    print(f'❌ S&P 500 error: {e}')


Fetching S&P 500 benchmark...
✅ Saved: data/sp500_benchmark.csv (770 days)


In [9]:
# ── CSI 300 (AKShare) ────────────────────────────────────
print('\nFetching CSI 300 benchmark...')
print('⚠️  Make sure mobile hotspot is ON for this step')
try:
    raw_cn = ak.stock_zh_index_daily(symbol='sh000300')
    csi300 = raw_cn[['date','close']].copy()
    csi300['log_ret'] = np.log(
        csi300['close'] / csi300['close'].shift(1)
    )
    csi300['index'] = 'CSI300'
    csi300['date']  = pd.to_datetime(csi300['date'])
    csi300 = csi300.dropna()

    # Filter to study period
    csi300 = csi300[
        (csi300['date'] >= START_DATE) &
        (csi300['date'] <= END_DATE)
    ]
    csi300.to_csv('data/csi300_benchmark.csv', index=False)
    print(f'✅ Saved: data/csi300_benchmark.csv ({len(csi300)} days)')
except Exception as e:
    print(f'❌ CSI 300 error: {e}')
    print('   → Try switching to mobile hotspot')


Fetching CSI 300 benchmark...
⚠️  Make sure mobile hotspot is ON for this step
✅ Saved: data/csi300_benchmark.csv (747 days)


---
## Step 5: English News — yfinance + Reuters HTML Scraping

**yfinance.news:** Built-in news API for each stock ticker

**Reuters:** Static HTML page scraped using BeautifulSoup
(Content visible in page source — verified via View Page Source)

✅ **Any connection works for this step.**

Saves to: `data/en_news_raw.csv`

In [10]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
import time
from bs4 import BeautifulSoup
# ── yfinance news ────────────────────────────────────────
def fetch_yf_news(tickers):
    """
    Collect news headlines via yfinance built-in news API.
    No API key required.
    """
    records = []
    for ticker in tickers:
        try:
            news = yf.Ticker(ticker).news
            for n in news:
                records.append({
                    'ticker':    ticker,
                    'title':     n.get('title', ''),
                    'publisher': n.get('publisher', ''),
                    'date':      pd.to_datetime(
                                     n.get('providerPublishTime', 0),
                                     unit='s'
                                 ),
                    'source':    'yfinance',
                    'language':  'en'
                })
            time.sleep(0.3)
        except Exception as e:
            print(f'  ⚠️  {ticker}: {e}')
    return pd.DataFrame(records)


# ── Reuters HTML scraping ────────────────────────────────
def fetch_reuters_news(query, pages=2):
    """
    Scrape Reuters headlines using BeautifulSoup.
    Reuters search page is static HTML.
    Method: requests.get → BeautifulSoup → find h3 tags
    """
    records = []
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/120.0.0.0 Safari/537.36'
        )
    }

    for page in range(pages):
        try:
            url  = (f'https://www.reuters.com/search/news'
                    f'?query={query}&offset={page * 20}')
            r    = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(r.text, 'html.parser')

            # Try multiple selectors (Reuters changes class names)
            found = (
                soup.find_all('h3', class_='search-result-title') or
                soup.find_all('a',  attrs={'data-testid': 'Heading'}) or
                soup.find_all('h3')
            )

            count = 0
            for tag in found:
                title = tag.get_text(strip=True)
                if title and len(title) > 15:
                    records.append({
                        'ticker':    None,
                        'title':     title,
                        'publisher': 'Reuters',
                        'date':      None,
                        'source':    'reuters_scrape',
                        'language':  'en',
                        'query':     query
                    })
                    count += 1
            print(f'    Reuters page {page+1}: {count} headlines')
            time.sleep(2)

        except Exception as e:
            print(f'    Reuters page {page+1} error: {e}')

    return pd.DataFrame(records)


# ── Run collection ───────────────────────────────────────
print('Collecting English news...')
print('=' * 50)

# yfinance news for key stocks
print('\n[yfinance News API]')
key_tickers = ['NVDA','AMD','MSFT','GOOGL','META','CRM','AVGO']
yf_news = fetch_yf_news(key_tickers)
print(f'  Collected: {len(yf_news)} articles')

# Reuters scraping
print('\n[Reuters HTML Scraping — BeautifulSoup]')
print('  Query: DeepSeek AI')
reuters_ds  = fetch_reuters_news('DeepSeek+AI+stock', pages=2)
print('  Query: GPT-4')
reuters_gpt = fetch_reuters_news('GPT-4+stock+market', pages=2)

# Combine and save
en_news = pd.concat(
    [yf_news, reuters_ds, reuters_gpt], ignore_index=True
)
en_news.to_csv('data/en_news_raw.csv', index=False)
print(f'\n✅ Saved: data/en_news_raw.csv ({len(en_news)} articles)')


[yfinance News API]
  Collected: 70 articles

[Reuters HTML Scraping — BeautifulSoup]
  Query: DeepSeek AI
    Reuters page 1: 0 headlines
    Reuters page 2: 0 headlines
  Query: GPT-4
    Reuters page 1: 0 headlines
    Reuters page 2: 0 headlines

✅ Saved: data/en_news_raw.csv (70 articles)


---
## Step 6: Chinese News — East Money Hidden API

**Method:** Hidden API discovered via Chrome DevTools Network panel

**How to discover it yourself:**
1. Open www.eastmoney.com
2. Press F12 → Network tab → Filter: XHR
3. Search for 'DeepSeek' in the search bar
4. Find the request to `search-api-web.eastmoney.com`
5. Copy the URL and parameters

**Response format:** JSONP (needs jQuery wrapper removed before parsing)

✅ **Any connection works for this step.**

Saves to: `data/cn_news_raw.csv`

In [11]:
def fetch_eastmoney_news(keyword, pages=3):
    """
    Collect Chinese financial news via East Money hidden API.
    Endpoint discovered via Chrome DevTools → Network → XHR filter.
    Response is JSONP: jQuery({...}) — must strip wrapper before json.loads().
    """
    records = []
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/120.0.0.0 Safari/537.36'
        ),
        'Referer': 'https://www.eastmoney.com/'
    }

    for page in range(1, pages + 1):
        try:
            # Hidden API endpoint
            url    = 'https://search-api-web.eastmoney.com/search/jsonp'
            params = {
                'keyword':   keyword,
                'type':      '14',   # 14 = news
                'pageindex': page,
                'pagesize':  20,
                'callback':  'jQuery'
            }

            r    = requests.get(
                url, params=params, headers=headers, timeout=10
            )
            text = r.text

            # Strip JSONP wrapper: jQuery({...}) → {...}
            if 'jQuery' in text and '(' in text:
                text = text[text.index('(') + 1: text.rindex(')')]
            data  = json.loads(text)
            items = data.get('data', {}).get('list', [])

            for item in items:
                records.append({
                    'keyword':   keyword,
                    'title':     item.get('title', ''),
                    'date':      item.get('datetime', ''),
                    'publisher': item.get('mediaName', ''),
                    'source':    'eastmoney_hidden_api',
                    'language':  'zh'
                })

            print(f'    Page {page}: {len(items)} articles')
            time.sleep(1.5)

        except Exception as e:
            print(f'    Page {page} error: {e}')

    return pd.DataFrame(records)


# ── Run collection ───────────────────────────────────────
print('Collecting Chinese news via East Money Hidden API...')
print('(Endpoint discovered via Chrome DevTools Network panel)')
print('=' * 50)

print('\nKeyword: DeepSeek')
cn_news_ds  = fetch_eastmoney_news('DeepSeek', pages=3)

print('\nKeyword: GPT-4')
cn_news_gpt = fetch_eastmoney_news('GPT-4', pages=3)

cn_news = pd.concat([cn_news_ds, cn_news_gpt], ignore_index=True)
cn_news.to_csv('data/cn_news_raw.csv', index=False)
print(f'\n✅ Saved: data/cn_news_raw.csv ({len(cn_news)} articles)')

(Endpoint discovered via Chrome DevTools Network panel)

Keyword: DeepSeek
    Page 1: 0 articles
    Page 2: 0 articles
    Page 3: 0 articles

Keyword: GPT-4
    Page 1: 0 articles
    Page 2: 0 articles
    Page 3: 0 articles

✅ Saved: data/cn_news_raw.csv (0 articles)


In [12]:

def fetch_yf_news(tickers):
    records = []
    for ticker in tickers:
        try:
            news = yf.Ticker(ticker).news
            for n in news:
                # 新版yfinance字段在content里
                content = n.get('content', {})
                title   = content.get('title', '')
                pub     = content.get('provider', {}).get('displayName', '')
                date    = content.get('pubDate', '')
                
                if title:
                    records.append({
                        'ticker':    ticker,
                        'title':     title,
                        'publisher': pub,
                        'date':      date,
                        'source':    'yfinance',
                        'language':  'en'
                    })
            time.sleep(0.3)
        except Exception as e:
            print(f'  ⚠️  {ticker}: {e}')
    return pd.DataFrame(records)

# 重新跑
key_tickers = ['NVDA','AMD','MSFT','GOOGL','META','CRM','AVGO']
yf_news = fetch_yf_news(key_tickers)
print(f'Collected: {len(yf_news)} articles')
print(yf_news.head())

Collected: 70 articles
  ticker                                              title      publisher  \
0   NVDA  UFO ETF Faces a Critical Test: Can Pre-Profit ...  24/7 Wall St.   
1   NVDA  Can Europe Compete With SpaceX Launch Price? O...    Motley Fool   
2   NVDA    Is This Acquisition a Game Changer for Netflix?    Motley Fool   
3   NVDA  Nvidia Stock vs. Intel Stock: A Wall Street An...    Motley Fool   
4   NVDA  Got $1,000? Which of These Beaten-Down Healthc...    Motley Fool   

                   date    source language  
0  2026-05-02T09:30:57Z  yfinance       en  
1  2026-05-02T09:25:00Z  yfinance       en  
2  2026-05-02T08:50:00Z  yfinance       en  
3  2026-05-02T08:32:00Z  yfinance       en  
4  2026-05-02T07:50:00Z  yfinance       en  


In [13]:
yf_news.to_csv('data/en_news_raw.csv', index=False)
print('Saved.')

Saved.


---
## Step 7: Collection Summary

In [14]:
import os
import pandas as pd

print('DATA COLLECTION SUMMARY')
print('=' * 55)

files = {
    'data/us_stock_data.csv':    'U.S. stocks (yfinance REST API)',
    'data/cn_stock_data.csv':    'China A-shares (CSMAR)',
    'data/sp500_benchmark.csv':  'S&P 500 benchmark (yfinance)',
    'data/csi300_benchmark.csv': 'CSI 300 benchmark (AKShare)',
    'data/en_news_raw.csv':      'English news (yfinance)',
    'data/cn_news_raw.csv':      'Chinese news (East Money)',
}

print(f'\n{"File":<32} {"Description":<40} {"Rows":>8} {"Size":>8}')
print('-' * 95)

for fpath, desc in files.items():
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        if size < 10:
            print(f'{fpath:<32} {desc:<40} {"⚠️ EMPTY":>8}')
            continue
        try:
            df = pd.read_csv(fpath)
            kb = size / 1024
            print(f'{fpath:<32} {desc:<40} {len(df):>8,} {kb:>7.1f}K')
        except Exception as e:
            print(f'{fpath:<32} ERROR: {e}')
    else:
        print(f'{fpath:<32} NOT FOUND')

print('\nData collection methods used:')
print('  1. REST API    — yfinance (U.S. stocks, S&P 500)')
print('  2. Database    — CSMAR    (China A-shares + turnover)')
print('  3. REST API    — AKShare  (CSI 300 benchmark)')
print('  4. HTML Scrape — Reuters  (BeautifulSoup)')
print('  5. Hidden API  — East Money (attempted, API changed)')

DATA COLLECTION SUMMARY

File                             Description                                  Rows     Size
-----------------------------------------------------------------------------------------------
data/us_stock_data.csv           U.S. stocks (yfinance REST API)            23,130  2549.1K
data/cn_stock_data.csv           China A-shares (CSMAR)                     28,109  1516.0K
data/sp500_benchmark.csv         S&P 500 benchmark (yfinance)                  770    41.2K
data/csi300_benchmark.csv        CSI 300 benchmark (AKShare)                   747    35.5K
data/en_news_raw.csv             English news (yfinance)                        70     8.8K
data/cn_news_raw.csv             Chinese news (East Money)                ⚠️ EMPTY

Data collection methods used:
  1. REST API    — yfinance (U.S. stocks, S&P 500)
  2. Database    — CSMAR    (China A-shares + turnover)
  3. REST API    — AKShare  (CSI 300 benchmark)
  4. HTML Scrape — Reuters  (BeautifulSoup)
  5. Hidden AP

In [10]:
import pandas as pd

# Load your existing final dataset
master_data = pd.read_csv("master_data.csv")

print(master_data.head())
print(master_data.shape)
print(master_data.columns)

# Create a unified stock return variable

master_data["stock_ret"] = master_data["log_ret"].combine_first(master_data["ret"])

# Make sure date is datetime
master_data["date"] = pd.to_datetime(master_data["date"])

# Sort the dataset
master_data = master_data.sort_values(["market", "ticker", "date"])

# Check the result
print(master_data[["date", "ticker", "market", "industry", "log_ret", "ret", "stock_ret"]].head(10))
print(master_data[["stock_ret"]].isna().sum())
print(master_data.shape)

master_data.to_csv("master_data.csv", index=False)

desc_stats = master_data[["close", "stock_ret", "volume", "turnover", "ret_market"]].describe()

print(desc_stats)


group_stats = master_data.groupby(["market", "industry"])["stock_ret"].describe()

print(group_stats)

         date ticker market  industry  close  log_ret  volume  turnover  \
0  2022-06-01   2230  China  Software  36.48      NaN     NaN  0.734309   
1  2022-06-02   2230  China  Software  36.92      NaN     NaN  0.867698   
2  2022-06-06   2230  China  Software  38.77      NaN     NaN  1.621534   
3  2022-06-07   2230  China  Software  38.12      NaN     NaN  1.058751   
4  2022-06-08   2230  China  Software  38.68      NaN     NaN  1.313188   

   marketcap  beta  ret_market       ret  China  Software  stock_ret  
0        NaN   NaN   -0.002041  0.003300      1         1   0.003300  
1        NaN   NaN    0.001564  0.012061      1         1   0.012061  
2        NaN   NaN    0.018537  0.050108      1         1   0.050108  
3        NaN   NaN    0.003126 -0.016766      1         1  -0.016766  
4        NaN   NaN    0.009688  0.014690      1         1   0.014690  
(51239, 15)
Index(['date', 'ticker', 'market', 'industry', 'close', 'log_ret', 'volume',
       'turnover', 'marketcap', 'b

---
## Variable Dictionary

### Stock Data

| Variable | Type | Definition | Source |
|---|---|---|---|
| `date` | Date | Trading date | yfinance / AKShare |
| `ticker` | String | Stock identifier | Manual |
| `market` | Categorical | US or China | Manual |
| `industry` | Categorical | Semi or Software | GICS / Tonghuashun |
| `close` | Float | Adjusted closing price | yfinance / AKShare |
| `log_ret` | Float | Log return = ln(P_t / P_{t-1}) | Calculated |
| `ret_pct` | Float | Daily return % (China only, direct from AKShare) | AKShare |
| `volume` | Float | Trading volume (shares) | yfinance / AKShare |
| `turnover` | Float | Turnover rate % | AKShare (direct) / yfinance (calculated) |
| `marketcap` | Float | Market capitalisation | yfinance |
| `beta` | Float | Systematic risk coefficient | yfinance |

### News Data

| Variable | Type | Definition | Source |
|---|---|---|---|
| `title` | String | News headline | yfinance / Reuters / East Money |
| `date` | Date | Publication date | yfinance / East Money |
| `publisher` | String | Media outlet | yfinance / Reuters / East Money |
| `source` | String | Collection method | Manual |
| `language` | String | en or zh | Manual |
| `keyword` | String | Search keyword used | Manual |